<a href="https://colab.research.google.com/github/chenchihwang/MNIST/blob/main/Chen_Chi_Hwang_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
from torch import optim
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader

In [ ]:
torch.randn(5).cuda()

tensor([-0.9310, -1.4913, -0.2605, -0.2542, -0.5743], device='cuda:0')

In [ ]:
train_data = datasets.MNIST('data', train = True, download = False, transform = transforms.ToTensor())
train, val = random_split(train_data, [55000,5000])
train_loader = DataLoader(train, batch_size = 32)
val_loader = DataLoader(val, batch_size = 12)

In [ ]:
model = nn.Sequential(
    nn.Linear(28*28, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

In [ ]:
class ResNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.l1 = nn.Linear(28*28, 64)
    self.l2 = nn.Linear(64, 64)
    self.l3 = nn.Linear(64, 10)
    self.do = nn.Dropout(0.1)

  def forward(self,x):
    h1 = nn.functional.relu(self.l1(x))
    h2 = nn.functional.relu(self.l2(h1))
    do = self.do(h2 + h1)
    logits = self.l3(do)
    return logits
model = ResNet().cuda()

In [ ]:
params = model.parameters()
optimiser = optim.SGD(params, lr = 1e-2)

In [ ]:
loss = nn.CrossEntropyLoss()

In [ ]:
nb_epochs = 20
for epoch in range(nb_epochs):
  losses = list()
  accuracies = list()
  for batch in train_loader:
    x, y = batch

    b = x.size(0)
    x = x.view(b, -1).cuda()

    # 1 forward prop
    l = model(x)

    # 2 compute loss ?
    J = loss(l, y.cuda())

    # 3 cleaning the gradients
    model.zero_grad()

    # 4 accumulate the partial derivatives of J with respect to params
    J.backward()

    # 5 step in the opposite direction of gradient
    optimiser.step()

    losses.append(J.item())
    accuracies.append(y.eq(l.detach().argmax(dim=1).cpu()).float().mean())

  print(f"Epoch {epoch + 1 }", end = ", ")
  print(f"training loss: {torch.tensor(losses).mean():.2f}", end = ", ")
  print(f"training acc.: {torch.tensor(accuracies).mean():.2f}")


  losses = list()
  accuracies = list()
  for batch in val_loader:
    x, y = batch

    b = x.size(0)
    x = x.view(b, -1).cuda()

    # 1 forward prop
    with torch.no_grad():
      l = model(x)

    # 2 compute loss ?
    J = loss(l, y.cuda())

    losses.append(J.item())
    accuracies.append(y.eq(l.detach().argmax(dim=1).cpu()).float().mean())

  print(f"Epoch {epoch + 1 }", end = ", ")
  print(f"validation loss: {torch.tensor(losses).mean():.2f}", end = ", ")
  print(f"validation acc.: {torch.tensor(accuracies).mean():.2f}")


Epoch 1, training loss: 0.84, training acc.: 0.77
Epoch 1, validation loss: 0.45, validation acc.: 0.87
Epoch 2, training loss: 0.37, training acc.: 0.90
Epoch 2, validation loss: 0.35, validation acc.: 0.90
Epoch 3, training loss: 0.30, training acc.: 0.91
Epoch 3, validation loss: 0.30, validation acc.: 0.91
Epoch 4, training loss: 0.26, training acc.: 0.92
Epoch 4, validation loss: 0.27, validation acc.: 0.92
Epoch 5, training loss: 0.23, training acc.: 0.93
Epoch 5, validation loss: 0.25, validation acc.: 0.93
Epoch 6, training loss: 0.21, training acc.: 0.94
Epoch 6, validation loss: 0.23, validation acc.: 0.94
Epoch 7, training loss: 0.19, training acc.: 0.95
Epoch 7, validation loss: 0.21, validation acc.: 0.94
Epoch 8, training loss: 0.18, training acc.: 0.95
Epoch 8, validation loss: 0.20, validation acc.: 0.94
Epoch 9, training loss: 0.16, training acc.: 0.95
Epoch 9, validation loss: 0.19, validation acc.: 0.94
Epoch 10, training loss: 0.15, training acc.: 0.96
Epoch 10, val